# Training FolioGPT Model

- Base Model: Meta-Llama-3.1-8B-Instruct
- Optimization: LoRA (Low-Rank Adaptation) for parameter-efficient fine-tuning
- Quantization: 4-bit quantization to reduce memory usage
- Context Length: 8192 tokens with RoPE scaling

### Note
Training is performed on https://lightning.ai/  using an A100 GPU (Few components are not loaded due to that)

In [2]:
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
from torch import __version__; from packaging.version import Version as V
!pip install --no-deps trl peft accelerate bitsandbytes triton

## Import all necessary libraries and models

In [1]:
from unsloth import FastLanguageModel
import torch

# Model configuration
max_seq_length = 8192
dtype = None
load_in_4bit = True

# List of 4bit pre-quantized models supported by Unsloth
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit",
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3-mini-4k-instruct",
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",
]

# Load the base model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.11: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.25 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.


### We now add LoRA adapters

In [2]:
# Enable FastLanguageModel's optimized generation
FastLanguageModel.for_inference(model)  # Enable inference mode

# Test generation
prompt = "Explain quantum computing in simple terms:"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    use_cache=False,  # Enable KV cache for faster generation
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPREC

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Explain quantum computing in simple terms: A beginner's guide
Quantum computing is a new and exciting field that has been gaining attention in recent years. But what exactly is it, and how does it work? In this article, we'll break down the basics of quantum computing in simple terms, so you can understand the concept without needing a Ph.D. in physics.
**What is Quantum Computing?**

Quantum computing is a type of computing that uses the principles of quantum mechanics to perform calculations. Unlike classical computers, which use bits (0s and 1s) to process information, quantum computers use quantum bits or qubits.

**Qubits: The Building Blocks of Quantum Computing**

Qubits are the fundamental units of quantum computing. They're like the bits in classical computers, but with a twist. Qubits can exist in multiple states simultaneously, which allows them to process multiple possibilities at the same time.

Imagine a coin that can be either heads or tails. A classical bit would be eit

In [3]:
# Add LoRA adapters for efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,  
)

Unsloth 2026.3.11 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


## Prepare data in Alpaca format

In [4]:
import json
from datasets import Dataset

# Load the JSON data
with open('/teamspace/studios/this_studio/merged_data.json', 'r') as f:
    data = json.load(f)

# Convert JSON data to Dataset format
dataset_dict = {
    "instruction": [],
    "input": [],
    "output": []
}

for item in data:
    dataset_dict["instruction"].append(item["instruction"])
    dataset_dict["input"].append(item["input"])
    dataset_dict["output"].append(item["response"])

# Create Dataset object
dataset = Dataset.from_dict(dataset_dict)

# Define Alpaca prompt format
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN to prevent infinite generation
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Apply formatting to dataset
dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/2973 [00:00<?, ? examples/s]

## Set up training arguments and trainer

In [5]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 512,          # 🔥 reduce this (try 256 if needed)
    dataset_num_proc = 2,
    packing = True,                # 🔥 IMPORTANT (better memory usage)

    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,   # 🔥 huge improvement

        warmup_steps = 5,
        num_train_epochs = 1,
        learning_rate = 2e-4,

        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),

        logging_steps = 1,

        optim = "paged_adamw_8bit",  # 🔥 better than adamw_8bit

        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/2973 [00:00<?, ? examples/s]

In [15]:
# Display memory usage statistics
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-80GB. Max memory = 79.25 GB.
78.688 GB of memory reserved.


## Start training

In [16]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,973 | Num Epochs = 1 | Total steps = 372
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,2.133580
2,2.077845
3,2.003904
4,2.161857
5,2.333476
6,2.039611
7,1.580557
8,1.797341
9,1.658259
10,1.222762


In [17]:
# Display final memory and time statistics
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

856.3312 seconds used for training.
14.27 minutes used for training.
Peak reserved memory = 78.688 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 99.291 %.
Peak reserved memory for training % of max memory = 0.0 %.


## Save the model

In [ ]:
# Local saving
model.save_pretrained("FolioGPT") 
tokenizer.save_pretrained("FolioGPT")